# Extraction des représentations acoustiques — Corpus
**Projet : Caractérisation acoustique des langues tonales africaines**  
Master NLP Avancé — Phase 1.2

Ce notebook extrait et visualise pour chaque énoncé :
1. **Waveform** — signal temporel brut
2. **Spectrogramme STFT** — `n_fft=512`, `hop_length=160`, fenêtre de Hann
3. **Log-Mel Spectrogram** — 80 filtres Mel (configuration Whisper)
4. **MFCC** — 13 coefficients + Δ (delta) + ΔΔ (delta-delta)
5. **Contour F0** — extraction du pitch avec `librosa.pyin`

## 0. Installation des dépendances

In [ ]:
# Exécuter une seule fois
# !pip install librosa matplotlib numpy scipy pandas soundfile

## 1. Imports et paramètres acoustiques

Tous les paramètres sont centralisés ici — conformes aux spécifications du projet.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
from pathlib import Path
import csv

# ─────────────────────────────────────────────────────────────
#  PARAMÈTRES ACOUSTIQUES — NE PAS MODIFIER
# ─────────────────────────────────────────────────────────────

SR          = 16_000   # Fréquence d'échantillonnage cible (Hz)

# Paramètres STFT (spécifiés dans le sujet)
N_FFT       = 512      # Taille FFT → résolution fréq. = SR/N_FFT = 31.25 Hz
HOP_LENGTH  = 160      # Pas → résolution temporelle = HOP/SR = 10 ms
WIN_LENGTH  = 400      # Fenêtre d'analyse = 25 ms
WINDOW      = 'hann'   # Type de fenêtre (spécifié dans le sujet)

# Paramètres Log-Mel (configuration Whisper)
N_MELS      = 80       # Nombre de filtres Mel
F_MIN       = 0.0      # Fréquence minimale des filtres Mel
F_MAX       = SR / 2   # Fréquence maximale (Nyquist)

# Paramètres MFCC
N_MFCC      = 13       # Nombre de coefficients cepstraux

# Paramètres F0 / pyin
F0_MIN      = 75       # Hz — voix humaine minimum
F0_MAX      = 400      # Hz — voix humaine maximum

# Chemins
DOSSIER_AUDIO   = Path('./audios/fongbe')
DOSSIER_PLANCHES = Path('./planches')
CSV_CORPUS      = Path('./audios/fongbe/transcriptions_fongbe.csv')

DOSSIER_PLANCHES.mkdir(parents=True, exist_ok=True)

print('✅ Paramètres chargés')
print(f'   SR={SR} Hz | N_FFT={N_FFT} | HOP={HOP_LENGTH} | WIN={WIN_LENGTH} | Fenêtre={WINDOW}')
print(f'   N_MELS={N_MELS} | N_MFCC={N_MFCC} | F0=[{F0_MIN}–{F0_MAX}] Hz')
print(f'   Résolution temporelle  : {HOP_LENGTH/SR*1000:.1f} ms')
print(f'   Résolution fréquentielle: {SR/N_FFT:.2f} Hz')

## 2. Fonctions d'extraction

Chaque fonction correspond à une représentation du sujet.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  2.1  CHARGEMENT AUDIO
# ─────────────────────────────────────────────────────────────

def charger_audio(chemin):
    """
    Charge un fichier audio, le convertit en mono 16 kHz.
    Retourne : (signal numpy float32, sample_rate)
    """
    y, sr = librosa.load(str(chemin), sr=SR, mono=True)
    return y, sr


# ─────────────────────────────────────────────────────────────
#  2.2  SPECTROGRAMME STFT
# ─────────────────────────────────────────────────────────────

def extraire_stft(y, sr=SR):
    """
    Calcule le spectrogramme STFT en dB.
    
    Paramètres conformes au sujet :
      - n_fft=512  →  fenêtre de 32 ms à 16 kHz
      - hop_length=160  →  pas de 10 ms
      - window='hann'
    
    Retourne : (D_db, freqs_hz, times_sec)
    """
    D = librosa.stft(
        y,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=WINDOW
    )
    D_db    = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    freqs   = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    times   = librosa.frames_to_time(
        np.arange(D.shape[1]), sr=sr, hop_length=HOP_LENGTH
    )
    return D_db, freqs, times


# ─────────────────────────────────────────────────────────────
#  2.3  LOG-MEL SPECTROGRAM (config Whisper)
# ─────────────────────────────────────────────────────────────

def extraire_logmel(y, sr=SR):
    """
    Calcule le Log-Mel Spectrogram avec 80 filtres (config Whisper).
    
    C'est exactement la représentation utilisée en entrée de Whisper :
      80 filtres Mel, même fenêtrage que le STFT.
    
    Retourne : (mel_db, times_sec)
    """
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=WINDOW,
        n_mels=N_MELS,    # 80 filtres — config Whisper
        fmin=F_MIN,
        fmax=F_MAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    times  = librosa.frames_to_time(
        np.arange(mel.shape[1]), sr=sr, hop_length=HOP_LENGTH
    )
    return mel_db, times


# ─────────────────────────────────────────────────────────────
#  2.4  MFCC + DELTA + DELTA-DELTA
# ─────────────────────────────────────────────────────────────

def extraire_mfcc(y, sr=SR):
    """
    Calcule 13 MFCC + leurs dérivées première (Δ) et seconde (ΔΔ).
    
    - MFCC    : capture la forme spectrale (timbre)
    - Delta   : capture la dynamique temporelle (transitions)
    - DeltaDelta : capture l'accélération (utile pour les tons modulés)
    
    Retourne : dict avec 'mfcc', 'delta', 'delta2', 'times'
    """
    mfcc   = librosa.feature.mfcc(
        y=y, sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )
    delta  = librosa.feature.delta(mfcc, order=1)   # Δ
    delta2 = librosa.feature.delta(mfcc, order=2)   # ΔΔ
    times  = librosa.frames_to_time(
        np.arange(mfcc.shape[1]), sr=sr, hop_length=HOP_LENGTH
    )
    return {'mfcc': mfcc, 'delta': delta, 'delta2': delta2, 'times': times}


# ─────────────────────────────────────────────────────────────
#  2.5  CONTOUR F0 (pyin — probabiliste)
# ─────────────────────────────────────────────────────────────

def extraire_f0(y, sr=SR):
    """
    Extrait le contour de F0 via pyin (probabilistic YIN).
    
    pyin est plus robuste que yin sur :
      - les consonnes (évite les faux positifs)
      - les attaques consonantiques du fongbé (gb, kp)
      - les transitions ton Haut → Bas
    
    Retourne : (f0, voiced_flag, times_sec)
      - f0          : fréquences en Hz (NaN sur segments non voisés)
      - voiced_flag : booléen, True = segment voisé
      - times       : instants en secondes
    """
    f0, voiced_flag, _ = librosa.pyin(
        y,
        fmin=F0_MIN,
        fmax=F0_MAX,
        sr=sr,
        hop_length=HOP_LENGTH,
        frame_length=WIN_LENGTH
    )
    times = librosa.frames_to_time(
        np.arange(len(f0)), sr=sr, hop_length=HOP_LENGTH
    )
    return f0, voiced_flag, times


print('✅ Fonctions d\'extraction définies')

## 3. Fonctions de statistiques F0

Métriques demandées dans la **Phase 2** du projet.

In [ ]:
def stats_f0(f0, voiced, sr=SR):
    """
    Calcule les 5 métriques F0 demandées en Phase 2 :
      1. F0 moyenne (Hz)
      2. F0 écart-type (Hz)
      3. F0 range (Hz et demi-tons)
      4. Pente F0 moyenne (Hz/ms)
      5. Nombre de points d'inflexion par seconde
    """
    f0_voiced = f0[voiced & ~np.isnan(f0)]

    if len(f0_voiced) == 0:
        return {k: np.nan for k in [
            'f0_mean','f0_std','f0_range_hz','f0_range_st',
            'f0_slope_hz_per_ms','n_inflexions_per_sec','voiced_ratio'
        ]}

    # Range en demi-tons : 12 × log2(fmax/fmin)
    f0_range_hz = float(f0_voiced.max() - f0_voiced.min())
    f0_range_st = (12 * np.log2(f0_voiced.max() / f0_voiced.min())
                   if f0_voiced.min() > 0 else np.nan)

    # Pente moyenne (régression linéaire sur segments voisés)
    idx = np.where(voiced & ~np.isnan(f0))[0].astype(float)
    if len(idx) > 1:
        coef  = np.polyfit(idx * HOP_LENGTH / sr * 1000, f0_voiced, 1)
        pente = float(coef[0])
    else:
        pente = np.nan

    # Points d'inflexion : changements de signe de la dérivée première
    diff    = np.diff(f0_voiced)
    signe   = np.sign(diff)
    signe_nz = signe[signe != 0]
    n_inflex = int(np.sum(np.diff(signe_nz) != 0))
    duree   = len(f0) * HOP_LENGTH / sr
    inflex_per_sec = n_inflex / duree if duree > 0 else np.nan

    return {
        'f0_mean':             round(float(f0_voiced.mean()), 2),
        'f0_std':              round(float(f0_voiced.std()),  2),
        'f0_range_hz':         round(f0_range_hz, 2),
        'f0_range_st':         round(f0_range_st, 2) if not np.isnan(f0_range_st) else np.nan,
        'f0_slope_hz_per_ms':  round(pente, 4) if not np.isnan(pente) else np.nan,
        'n_inflexions_per_sec': round(inflex_per_sec, 2),
        'voiced_ratio':        round(float(voiced.mean()), 3),
    }


def stats_mfcc(mfcc_data):
    """Vecteur MFCC moyen (13 dim) utilisé pour le classifieur SVM Phase 2."""
    result = {}
    for i in range(N_MFCC):
        result[f'mfcc_{i+1:02d}_mean'] = round(float(mfcc_data['mfcc'][i].mean()), 4)
    result['delta_energy_mean']  = round(float(np.abs(mfcc_data['delta']).mean()), 4)
    result['delta2_energy_mean'] = round(float(np.abs(mfcc_data['delta2']).mean()), 4)
    return result


print('✅ Fonctions de statistiques définies')

## 4. Visualisation — planche complète par énoncé

Génère une planche avec les **5 représentations empilées** pour un énoncé.

In [ ]:
def tracer_planche(y, sr, meta, chemin_sortie=None, afficher=True):
    """
    Génère la planche complète des 5 représentations acoustiques.
    
    Paramètres
    ----------
    y             : signal audio numpy
    sr            : fréquence d'échantillonnage
    meta          : dict avec id, transcription_ortho, traduction_fr, locuteur
    chemin_sortie : chemin PNG de sortie (None = ne pas sauvegarder)
    afficher      : afficher inline dans le notebook
    """
    duree = librosa.get_duration(y=y, sr=sr)

    # ── Extractions ──────────────────────────────────────────────────────────
    D_db, freqs, t_stft   = extraire_stft(y, sr)
    mel_db, t_mel         = extraire_logmel(y, sr)
    mfcc_data             = extraire_mfcc(y, sr)
    f0, voiced, t_f0      = extraire_f0(y, sr)
    sf                    = stats_f0(f0, voiced, sr)

    # ── Mise en page ─────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 14))
    fig.patch.set_facecolor('#0D1117')

    titre = (
        f"{meta.get('id','?')}  ·  "
        f"{meta.get('transcription_ortho','')}  ·  "
        f"« {meta.get('traduction_fr','')} »  ·  "
        f"{meta.get('locuteur','')}  ·  "
        f"Durée : {duree:.2f}s"
    )
    fig.suptitle(titre, fontsize=10, color='white', y=0.98)

    gs = gridspec.GridSpec(5, 1, figure=fig, hspace=0.55,
                           top=0.94, bottom=0.05, left=0.08, right=0.97)
    axes = [fig.add_subplot(gs[i]) for i in range(5)]

    def _style(ax, titre_ax):
        ax.set_facecolor('#161B22')
        ax.tick_params(colors='white', labelsize=8)
        for sp in ax.spines.values():
            sp.set_edgecolor('#30363D')
        ax.set_title(titre_ax, color='white', fontsize=9,
                     loc='left', pad=4, fontweight='medium')
        ax.grid(axis='x', color='#30363D', linewidth=0.4, alpha=0.6)

    t_wave = np.linspace(0, duree, len(y))

    # ── 1. Waveform ───────────────────────────────────────────────────────────
    axes[0].plot(t_wave, y, color='#2D6FA3', linewidth=0.6, alpha=0.9)
    axes[0].axhline(0, color='#444C56', linewidth=0.5)
    axes[0].set_xlim(0, duree)
    axes[0].set_ylabel('Amplitude', color='white', fontsize=8)
    _style(axes[0], '1 · Waveform — signal temporel brut')

    # ── 2. STFT ───────────────────────────────────────────────────────────────
    img = librosa.display.specshow(
        D_db, sr=sr, hop_length=HOP_LENGTH,
        x_axis='time', y_axis='hz',
        ax=axes[1], cmap='magma'
    )
    plt.colorbar(img, ax=axes[1], format='%+2.0f dB', pad=0.01
                ).ax.yaxis.set_tick_params(color='white', labelsize=7)
    axes[1].set_ylim(0, 4000)
    axes[1].set_ylabel('Fréq. (Hz)', color='white', fontsize=8)
    _style(axes[1],
           f'2 · Spectrogramme STFT  (n_fft={N_FFT}, hop={HOP_LENGTH}, {WINDOW})')

    # ── 3. Log-Mel ────────────────────────────────────────────────────────────
    img = librosa.display.specshow(
        mel_db, sr=sr, hop_length=HOP_LENGTH,
        x_axis='time', y_axis='mel',
        ax=axes[2], cmap='inferno'
    )
    plt.colorbar(img, ax=axes[2], format='%+2.0f dB', pad=0.01
                ).ax.yaxis.set_tick_params(color='white', labelsize=7)
    axes[2].set_ylabel('Mel', color='white', fontsize=8)
    _style(axes[2], f'3 · Log-Mel Spectrogram  ({N_MELS} filtres — config Whisper)')

    # ── 4. MFCC + Δ + ΔΔ empilés ─────────────────────────────────────────────
    mfcc_stack = np.vstack([
        mfcc_data['mfcc'],
        mfcc_data['delta'],
        mfcc_data['delta2']
    ])
    vmax = np.percentile(np.abs(mfcc_stack), 98)
    img = librosa.display.specshow(
        mfcc_stack, sr=sr, hop_length=HOP_LENGTH,
        x_axis='time', ax=axes[3],
        cmap='RdBu_r', vmin=-vmax, vmax=vmax
    )
    plt.colorbar(img, ax=axes[3], pad=0.01
                ).ax.yaxis.set_tick_params(color='white', labelsize=7)
    # Séparateurs MFCC / Δ / ΔΔ
    for yline in [N_MFCC, N_MFCC * 2]:
        axes[3].axhline(yline - 0.5, color='white', linewidth=0.7,
                        linestyle='--', alpha=0.6)
    axes[3].set_yticks([N_MFCC//2, N_MFCC + N_MFCC//2, N_MFCC*2 + N_MFCC//2])
    axes[3].set_yticklabels(['MFCC', 'Δ', 'ΔΔ'], color='white', fontsize=8)
    _style(axes[3], f'4 · MFCC ({N_MFCC} coeff.) + Δ (delta) + ΔΔ (delta-delta)')

    # ── 5. Contour F0 ─────────────────────────────────────────────────────────
    # Zone voisée en fond
    axes[4].fill_between(t_f0, 0, F0_MAX,
                          where=voiced, alpha=0.12,
                          color='#27500A', label='voisé')
    # F0 voisé (trait plein vert)
    f0_plot = f0.copy()
    f0_plot[~voiced] = np.nan
    axes[4].plot(t_f0, f0_plot, color='#27500A',
                 linewidth=2.0, label='F0 voisé', zorder=3)
    # F0 interpolé (trait pointillé orange — vision globale du contour)
    f0_interp = pd.Series(f0).interpolate(method='linear').values
    axes[4].plot(t_f0, f0_interp, color='#E8593C',
                 linewidth=0.9, alpha=0.5, linestyle='--',
                 label='F0 interpolé')

    # Annotation statistiques F0
    annot = (
        f"moy={sf['f0_mean']:.0f} Hz  "
        f"σ={sf['f0_std']:.0f} Hz  "
        f"range={sf['f0_range_hz']:.0f} Hz "
        f"({sf['f0_range_st']:.1f} st)"
    )
    axes[4].text(0.01, 0.93, annot, transform=axes[4].transAxes,
                 color='white', fontsize=7.5, va='top',
                 bbox=dict(boxstyle='round,pad=0.3',
                           facecolor='#1C2128', edgecolor='#444C56', alpha=0.9))

    # Patron tonal
    patron = meta.get('ton_patron', '')
    if patron and patron not in ('NA', ''):
        axes[4].text(0.99, 0.93, f'Ton : {patron}',
                     transform=axes[4].transAxes,
                     color='#EF9F27', fontsize=10, va='top', ha='right',
                     fontweight='bold')

    axes[4].set_ylim(F0_MIN, F0_MAX)
    axes[4].set_xlim(0, duree)
    axes[4].set_ylabel('F0 (Hz)', color='white', fontsize=8)
    axes[4].set_xlabel('Temps (s)', color='white', fontsize=8)
    axes[4].legend(fontsize=7, loc='upper right',
                   facecolor='#1C2128', edgecolor='#444C56', labelcolor='white')
    _style(axes[4], '5 · Contour F0 — extraction pyin (probabiliste)')

    if chemin_sortie:
        Path(chemin_sortie).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(chemin_sortie), facecolor=fig.get_facecolor(),
                    bbox_inches='tight', dpi=150)
        print(f'   Planche sauvegardée → {chemin_sortie}')

    if afficher:
        plt.show()
    else:
        plt.close(fig)


print('✅ Fonction de visualisation définie')

## 5. Test sur un énoncé unique

Visualiser un seul fichier pour vérifier que tout fonctionne.

In [ ]:
# ── Changer ce chemin selon ton corpus ────────────────────────────────────────
FICHIER_TEST = DOSSIER_AUDIO / 'fon_001.wav'

META_TEST = {
    'id':                 'fon_001',
    'transcription_ortho':'àzɔ́n',
    'traduction_fr':      'maladie',
    'locuteur':           'L1_féminin',
    'ton_patron':         'H',
}

if FICHIER_TEST.exists():
    y, sr = charger_audio(FICHIER_TEST)
    print(f'Audio chargé : {len(y)/sr:.2f}s — {len(y)} échantillons à {sr} Hz')
    tracer_planche(
        y, sr,
        meta=META_TEST,
        chemin_sortie=DOSSIER_PLANCHES / 'fon_001_planche.png',
        afficher=True
    )
else:
    print(f'⚠️  Fichier introuvable : {FICHIER_TEST}')
    print('   Modifie FICHIER_TEST avec le chemin d\'un de tes fichiers audio.')

## 6. Planche comparative — paire minimale tonale

**Exemple : àzɔ́n (maladie, Ton H) vs àzɔ̀n (travail, Ton B)**

C'est la planche la plus importante pour le projet — elle montre visuellement comment le F0 distingue deux mots identiques segmentalement.

In [ ]:
def tracer_paire_minimale(meta_h, meta_b,
                          chemin_wav_h, chemin_wav_b,
                          chemin_sortie=None, afficher=True):
    """
    Planche comparative côte à côte :
      Gauche = Ton Haut (H) | Droite = Ton Bas (B)
      Lignes  = STFT | Log-Mel | F0
    """
    y_h, _ = charger_audio(chemin_wav_h)
    y_b, _ = charger_audio(chemin_wav_b)

    def _ext(y):
        D_db, _, _ = extraire_stft(y)
        mel_db, _  = extraire_logmel(y)
        f0, vd, tf = extraire_f0(y)
        return D_db, mel_db, f0, vd, tf

    D_h, mel_h, f0_h, vd_h, tf_h = _ext(y_h)
    D_b, mel_b, f0_b, vd_b, tf_b = _ext(y_b)

    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    fig.patch.set_facecolor('#0D1117')
    fig.suptitle(
        f"Paire minimale : {meta_h.get('id_paire','')}  ·  "
        f"« {meta_h.get('traduction_fr','')} » vs « {meta_b.get('traduction_fr','')} »",
        color='white', fontsize=11
    )
    plt.subplots_adjust(hspace=0.45, wspace=0.25,
                        top=0.92, bottom=0.06, left=0.08, right=0.97)

    couleurs = ['#E8593C', '#378ADD']
    labels   = [
        f"TON HAUT (H) — {meta_h.get('transcription_ortho','')}",
        f"TON BAS  (B) — {meta_b.get('transcription_ortho','')}"
    ]
    donnees  = [
        (D_h, mel_h, f0_h, vd_h, tf_h, librosa.get_duration(y=y_h, sr=SR)),
        (D_b, mel_b, f0_b, vd_b, tf_b, librosa.get_duration(y=y_b, sr=SR)),
    ]

    for col in range(2):
        couleur = couleurs[col]
        D_db, mel_db, f0, vd, tf, dur = donnees[col]

        axes[0, col].set_title(labels[col], color=couleur,
                                fontsize=10, fontweight='bold')

        # STFT
        librosa.display.specshow(D_db, sr=SR, hop_length=HOP_LENGTH,
                                  x_axis='time', y_axis='hz',
                                  ax=axes[0, col], cmap='magma')
        axes[0, col].set_ylim(0, 4000)

        # Log-Mel
        librosa.display.specshow(mel_db, sr=SR, hop_length=HOP_LENGTH,
                                  x_axis='time', y_axis='mel',
                                  ax=axes[1, col], cmap='inferno')

        # F0
        f0_plot = f0.copy(); f0_plot[~vd] = np.nan
        axes[2, col].fill_between(tf, 0, F0_MAX, where=vd,
                                   alpha=0.12, color='#27500A')
        axes[2, col].plot(tf, f0_plot, color=couleur, linewidth=2.2, zorder=3)
        f0_interp = pd.Series(f0).interpolate('linear').values
        axes[2, col].plot(tf, f0_interp, color=couleur,
                           linewidth=0.8, alpha=0.4, linestyle='--')

        # Annotation stats F0
        sf_v = stats_f0(f0, vd, SR)
        axes[2, col].text(
            0.02, 0.93,
            f"moy={sf_v['f0_mean']:.0f} Hz  range={sf_v['f0_range_hz']:.0f} Hz ({sf_v['f0_range_st']:.1f} st)",
            transform=axes[2, col].transAxes, color='white',
            fontsize=7.5, va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#1C2128',
                      edgecolor='#444C56', alpha=0.9)
        )
        axes[2, col].set_ylim(F0_MIN, F0_MAX)
        axes[2, col].set_xlim(0, dur)
        axes[2, col].set_xlabel('Temps (s)', color='white', fontsize=8)

        # Style commun
        for ax in axes[:, col]:
            ax.set_facecolor('#161B22')
            ax.tick_params(colors='white', labelsize=7)
            for sp in ax.spines.values():
                sp.set_edgecolor('#30363D')

    for row, lbl in enumerate(['STFT', 'Log-Mel (80 filtres)', 'F0 pyin']):
        axes[row, 0].set_ylabel(lbl, color='white', fontsize=9, fontweight='medium')

    if chemin_sortie:
        Path(chemin_sortie).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(chemin_sortie), facecolor=fig.get_facecolor(),
                    bbox_inches='tight', dpi=150)
        print(f'   Planche paire → {chemin_sortie}')

    if afficher:
        plt.show()
    else:
        plt.close(fig)


# ── Exemple d'utilisation ─────────────────────────────────────────────────────
tracer_paire_minimale(
    meta_h={'id_paire':'pair_fon_01', 'transcription_ortho':'àzɔ́n',
            'traduction_fr':'maladie'},
    meta_b={'id_paire':'pair_fon_01', 'transcription_ortho':'àzɔ̀n',
            'traduction_fr':'travail'},
    chemin_wav_h=DOSSIER_AUDIO / 'fon_001.wav',
    chemin_wav_b=DOSSIER_AUDIO / 'fon_002.wav',
    chemin_sortie=DOSSIER_PLANCHES / 'comp_pair_fon_01.png',
    afficher=True
)

## 7. Pipeline complet — tout le corpus

Traite tous les énoncés du CSV et exporte les planches + le fichier `features_fongbe.csv` pour la Phase 2.

In [ ]:
def pipeline_corpus(csv_path, dossier_audio, dossier_planches,
                    csv_features='features_fongbe.csv',
                    n_max=None, afficher=False):
    """
    Traite tout le corpus :
      - Génère une planche PNG par énoncé
      - Génère les planches comparatives des paires minimales
      - Exporte features_fongbe.csv pour le classifieur SVM (Phase 2)
    """
    with open(csv_path, encoding='utf-8-sig') as f:
        corpus = [r for r in csv.DictReader(f)]

    if n_max:
        corpus = corpus[:n_max]

    print(f'\n{"═"*60}')
    print(f'  Pipeline acoustique — {len(corpus)} énoncés')
    print(f'{"═"*60}\n')

    rows_features = []
    audios_cache  = {}

    for i, meta in enumerate(corpus, 1):
        eid       = meta.get('id', f'fon_{i:03d}')
        wav_path  = Path(dossier_audio) / meta.get('fichier_audio', f'{eid}.wav')

        print(f'[{i:>3}/{len(corpus)}]  {eid}  '
              f'({meta.get("transcription_ortho","")[:30]})')

        if not wav_path.exists():
            print(f'   ⚠️  Fichier manquant : {wav_path}')
            continue

        y, sr = charger_audio(wav_path)
        audios_cache[eid] = (y, sr)

        # Planche individuelle
        tracer_planche(
            y, sr, meta,
            chemin_sortie=Path(dossier_planches) / f'{eid}_planche.png',
            afficher=afficher
        )

        # Features pour Phase 2
        mfcc_data    = extraire_mfcc(y, sr)
        f0, vd, _    = extraire_f0(y, sr)
        row = {
            'id': eid, 'langue': 'fongbe',
            'locuteur':       meta.get('locuteur',''),
            'ton_patron':     meta.get('ton_patron',''),
            'paire_minimale': meta.get('paire_minimale','non'),
            'id_paire':       meta.get('id_paire',''),
            'duree_sec':      round(librosa.get_duration(y=y, sr=sr), 3),
            **stats_f0(f0, vd, sr),
            **stats_mfcc(mfcc_data),
        }
        rows_features.append(row)

    # Planches comparatives paires minimales
    print(f'\n  Génération des planches paires minimales…')
    paires = {}
    for meta in corpus:
        pid = meta.get('id_paire','')
        if meta.get('paire_minimale') == 'oui' and pid:
            paires.setdefault(pid, []).append(meta)

    for pid, membres in paires.items():
        if len(membres) != 2:
            continue
        m_sorted = sorted(membres,
                          key=lambda m: 0 if m.get('ton_patron','') == 'H' else 1)
        id_h, id_b = m_sorted[0]['id'], m_sorted[1]['id']
        if id_h not in audios_cache or id_b not in audios_cache:
            continue
        wav_h = Path(dossier_audio) / m_sorted[0]['fichier_audio']
        wav_b = Path(dossier_audio) / m_sorted[1]['fichier_audio']
        tracer_paire_minimale(
            m_sorted[0], m_sorted[1], wav_h, wav_b,
            chemin_sortie=Path(dossier_planches) / f'comp_{pid}.png',
            afficher=afficher
        )

    # Export CSV features
    df = pd.DataFrame(rows_features)
    df.to_csv(csv_features, index=False, encoding='utf-8-sig')

    print(f'\n{"═"*60}')
    print(f'  ✅  {len(rows_features)} énoncés traités')
    print(f'  📁  Planches  → {dossier_planches}/')
    print(f'  📄  Features  → {csv_features}')
    print(f'{"═"*60}\n')
    return df


# ── Lancer le pipeline sur tout le corpus ─────────────────────────────────────
if CSV_CORPUS.exists():
    df_features = pipeline_corpus(
        csv_path        = CSV_CORPUS,
        dossier_audio   = DOSSIER_AUDIO,
        dossier_planches= DOSSIER_PLANCHES,
        csv_features    = 'features_fongbe.csv',
        afficher        = False   # True = afficher inline (lent)
    )
    df_features.head()
else:
    print(f'⚠️  CSV introuvable : {CSV_CORPUS}')
    print('   Lancer d\'abord generer_fongbe.py pour créer le corpus.')

## 8. Vérification des paramètres

Confirme que chaque représentation utilise bien les paramètres du sujet.

In [ ]:
print('Vérification des paramètres :')
print()
print(f'  Représentation       Paramètre            Valeur     Sujet')
print(f'  {"─"*65}')
print(f'  STFT                 n_fft                {N_FFT:<10} ✅ 512')
print(f'  STFT                 hop_length           {HOP_LENGTH:<10} ✅ 160')
print(f'  STFT                 window               {WINDOW:<10} ✅ Hann')
print(f'  Log-Mel              n_mels               {N_MELS:<10} ✅ 80 (Whisper)')
print(f'  MFCC                 n_mfcc               {N_MFCC:<10} ✅ 13')
print(f'  MFCC                 delta order          1          ✅ Δ')
print(f'  MFCC                 delta order          2          ✅ ΔΔ')
print(f'  F0                   méthode              pyin       ✅ pyin')
print(f'  F0                   fmin                 {F0_MIN:<10} ✅ 75 Hz')
print(f'  F0                   fmax                 {F0_MAX:<10} ✅ 400 Hz')
print()
print(f'  Résolution temporelle  : {HOP_LENGTH/SR*1000:.1f} ms/trame')
print(f'  Résolution fréquentielle: {SR/N_FFT:.2f} Hz/bin')
print(f'  Durée fenêtre d\'analyse : {WIN_LENGTH/SR*1000:.1f} ms')